# Step 8 — Confusion Matrix: Deep Dive
### Credit Card Underwriting Pipeline

---

## The Confusion Matrix Explained

> **The confusion matrix is the most honest report card a classifier can have.**
> Every prediction falls into one of 4 cells:

```
                      PREDICTED
                  Declined    Approved
ACTUAL  Declined |    TN    |    FP   |
        Approved |    FN    |    TP   |
```

| Cell | Name | Meaning | Business Impact |
|------|------|---------|----------------|
| **TN** | True Negative | Predicted Declined, Actually Declined | Correctly rejected a bad applicant |
| **FP** | False Positive | Predicted Approved, Actually Declined | **APPROVED A BAD APPLICANT** — credit loss risk |
| **FN** | False Negative | Predicted Declined, Actually Approved | **REJECTED A GOOD APPLICANT** — lost revenue |
| **TP** | True Positive | Predicted Approved, Actually Approved | Correctly approved a good applicant |

---

## The Business Cost of Each Error Type

> **In credit underwriting, not all errors are equal:**
>
> - **False Positive (FP)** = we approve someone who will default
>   → The bank loses the entire outstanding balance when they default
>   → Financial cost: potentially thousands of dollars per default
>
> - **False Negative (FN)** = we decline someone who would have repaid perfectly
>   → The bank loses the interest income and fee revenue
>   → Opportunity cost: maybe $200–500 in annual fees/interest lost
>
> **Implication:** In credit risk, FP is usually more costly than FN.
> The model's threshold may need to be adjusted to reduce FP even at the cost of more FN.

---

## Key Metrics Derived from the Confusion Matrix

| Metric | Formula | Meaning |
|--------|---------|---------|
| **Accuracy** | (TP+TN)/(TP+TN+FP+FN) | Overall % correct |
| **Precision** | TP/(TP+FP) | Of all predicted Approved, how many actually repaid? |
| **Recall (Sensitivity)** | TP/(TP+FN) | Of all actual Approved, how many did we correctly approve? |
| **Specificity** | TN/(TN+FP) | Of all actual Declined, how many did we correctly decline? |
| **F1-Score** | 2×Precision×Recall/(P+R) | Harmonic mean of Precision and Recall |

> **Why not just use Accuracy?**
> If 65% of applicants are Approved and we always predict Approved, accuracy = 65%.
> But Precision for Declined = 0% — we never declined anyone!
> Accuracy hides this disaster. Precision, Recall, and F1 reveal it.


In [ ]:
# ── Full pipeline re-run ─────────────────────────────────────────────────────
import warnings, numpy as np, pandas as pd, time
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
    classification_report, roc_auc_score, accuracy_score,
    precision_score, recall_score, f1_score)
from imblearn.over_sampling import SMOTE
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid')
SEED=42

DATA_PATH='cc_underwriting_5k_stratified11.csv'
IGNORE_COLS=['applicant_id','target_approved','target_credit_limit_assigned']
df_raw=pd.read_csv(DATA_PATH)
numeric_cols=[c for c in df_raw.select_dtypes(include='number').columns if c not in IGNORE_COLS]
cat_cols=[c for c in df_raw.select_dtypes(include=['object','string']).columns if c not in IGNORE_COLS]
def missing_report(df):
    m=df.isnull().sum();p=m/len(df)*100
    r=pd.DataFrame({'missing_count':m,'missing_pct':p.round(2)})
    return r[r['missing_count']>0].sort_values('missing_pct',ascending=False)
miss=missing_report(df_raw)
drop_cols=miss[miss['missing_pct']>50].index.tolist()
df=df_raw.drop(columns=drop_cols)
numeric_cols=[c for c in numeric_cols if c in df.columns]
cat_cols=[c for c in cat_cols if c in df.columns]
for col in numeric_cols:
    if df[col].isnull().sum()>0: df[col].fillna(df[col].median(),inplace=True)
for col in cat_cols:
    if df[col].isnull().sum()>0: df[col].fillna(df[col].mode()[0],inplace=True)
df['target']=(df['target_approved']=='Yes').astype(int)
df_fe=df.copy();eps=1e-6
df_fe['income_to_limit_ratio']=df_fe['annual_income']/(df_fe['requested_credit_limit']+eps)
df_fe['bureau_score_mean']=df_fe[['fico_score','equifax_score','experian_score','transunion_score']].mean(axis=1)
df_fe['bureau_score_std']=df_fe[['fico_score','equifax_score','experian_score','transunion_score']].std(axis=1)
df_fe['derogatory_x_dti']=df_fe['derogatory_marks_count']*df_fe['debt_to_income_ratio']
df_fe['monthly_net_income']=df_fe['monthly_income']-df_fe['total_monthly_expenses']
for col in ['annual_income','net_worth','total_assets','requested_credit_limit']:
    if col in df_fe.columns: df_fe[col+'_log']=np.log1p(np.clip(df_fe[col],0,None))
le=LabelEncoder()
cat_for_model=[c for c in cat_cols if df_fe[c].nunique()<=30]
for col in cat_for_model: df_fe[col+'_enc']=le.fit_transform(df_fe[col].astype(str))
FINAL_FEATURES=[c for c in numeric_cols+['income_to_limit_ratio','bureau_score_mean',
    'bureau_score_std','derogatory_x_dti','monthly_net_income']+
    [c+'_log' for c in ['annual_income','net_worth','total_assets','requested_credit_limit'] if c+'_log' in df_fe.columns]+
    [c+'_enc' for c in cat_for_model] if c in df_fe.columns]
X=df_fe[FINAL_FEATURES].fillna(0);y=df_fe['target']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=SEED,stratify=y)
X_train_sm,y_train_sm=SMOTE(random_state=SEED,k_neighbors=5).fit_resample(X_train,y_train)
scaler=StandardScaler()
X_train_sc=scaler.fit_transform(X_train_sm);X_test_sc=scaler.transform(X_test)
best_rf=RandomForestClassifier(n_estimators=200,max_depth=15,min_samples_leaf=2,
    class_weight='balanced',random_state=SEED,n_jobs=-1)
best_rf.fit(X_train_sc,y_train_sm)
y_pred=best_rf.predict(X_test_sc)
y_pred_proba=best_rf.predict_proba(X_test_sc)[:,1]
print('Pipeline complete. Model trained and predictions made.')

## 8.1 — The Confusion Matrix (Standard Threshold = 0.5)

> By default, the model predicts "Approved" if `predict_proba >= 0.5`.
> This is the standard threshold, but it can be adjusted (see Section 8.3).


In [ ]:
# Compute the confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Extract the 4 cells
TN = cm[0, 0]   # True Negative:  Predicted Declined, Actually Declined
FP = cm[0, 1]   # False Positive: Predicted Approved, Actually Declined (BAD — credit loss)
FN = cm[1, 0]   # False Negative: Predicted Declined, Actually Approved (BAD — lost revenue)
TP = cm[1, 1]   # True Positive:  Predicted Approved, Actually Approved

print('Confusion Matrix:')
print(f'                Predicted Declined  Predicted Approved')
print(f'Actual Declined       {TN:>5}              {FP:>5}')
print(f'Actual Approved       {FN:>5}              {TP:>5}')
print()
print(f'TN (Correct declines)  : {TN:>5}  — Correctly identified bad applicants')
print(f'FP (False approvals)   : {FP:>5}  — Approved someone who should be declined (credit risk!)')
print(f'FN (False declines)    : {FN:>5}  — Declined someone who would have repaid (lost revenue)')
print(f'TP (Correct approvals) : {TP:>5}  — Correctly identified good applicants')

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Standard confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Declined', 'Approved'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix\n(Standard Threshold = 0.50)', fontsize=13, fontweight='bold')

# Annotated version with business meaning
annot = np.array([
    [f'TN\n{TN}\nCorrect\nDecline', f'FP\n{FP}\nFalse\nApproval'],
    [f'FN\n{FN}\nFalse\nDecline', f'TP\n{TP}\nCorrect\nApproval']
])
im = axes[1].imshow([[TN,FP],[FN,TP]], cmap='RdYlGn', aspect='auto')
axes[1].set_xticks([0,1]); axes[1].set_xticklabels(['Predicted Declined','Predicted Approved'])
axes[1].set_yticks([0,1]); axes[1].set_yticklabels(['Actual Declined','Actual Approved'])
axes[1].set_title('Confusion Matrix\n(Business Annotated)', fontsize=13, fontweight='bold')
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, annot[i,j], ha='center', va='center',
                     fontsize=11, fontweight='bold',
                     color='black' if [TN,FP,FN,TP][i*2+j] > max(TN,FP,FN,TP)*0.5 else 'white')

plt.tight_layout()
plt.show()

## 8.2 — Derived Metrics from the Confusion Matrix

> These metrics tell the full story of model performance.
> **No single metric is sufficient** — always look at all of them together.


In [ ]:
# Compute all derived metrics from the confusion matrix
accuracy    = (TP + TN) / (TP + TN + FP + FN)
precision   = TP / (TP + FP)          # of predicted approvals, how many were correct?
recall      = TP / (TP + FN)          # of actual approvals, how many did we catch?
specificity = TN / (TN + FP)          # of actual declines, how many did we correctly decline?
f1          = 2 * precision * recall / (precision + recall)
fpr         = FP / (FP + TN)          # false positive rate

print('=' * 60)
print('     CONFUSION MATRIX DERIVED METRICS')
print('=' * 60)
print()
print(f'  Accuracy              : {accuracy:.4f}  ({accuracy:.1%})')
print(f'    -> Overall % of correct predictions')
print()
print(f'  Precision (Approved)  : {precision:.4f}  ({precision:.1%})')
print(f'    -> Of all we said "Approved", this % actually repaid')
print()
print(f'  Recall (Sensitivity)  : {recall:.4f}  ({recall:.1%})')
print(f'    -> Of all good applicants, we correctly approved this %')
print()
print(f'  Specificity           : {specificity:.4f}  ({specificity:.1%})')
print(f'    -> Of all bad applicants, we correctly declined this %')
print()
print(f'  False Positive Rate   : {fpr:.4f}  ({fpr:.1%})')
print(f'    -> Of all bad applicants, we WRONGLY approved this % (credit risk!)')
print()
print(f'  F1-Score              : {f1:.4f}')
print(f'    -> Harmonic mean of Precision and Recall (balance of both)')
print()

# Full classification report
print('CLASSIFICATION REPORT:')
print(classification_report(y_test, y_pred, target_names=['Declined', 'Approved']))

## 8.3 — Threshold Adjustment: Trading FP for FN

> **The 0.5 threshold is not sacred.** By moving it, we can trade one type of error for another.
>
> - **Raise threshold** (e.g., 0.7): Only approve when very confident → fewer FP (less credit risk)
>   but more FN (more good applicants wrongly declined)
> - **Lower threshold** (e.g., 0.3): Approve when moderately confident → more approvals,
>   more TP, but also more FP (higher default risk)
>
> **Business context drives the threshold choice:**
> - Conservative bank (high credit losses recently) → raise threshold
> - Growth-focused bank (wants more customers) → lower threshold


In [ ]:
# Show how the confusion matrix changes at different thresholds
thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]
threshold_results = []

print(f'{"Threshold":>10} {"TN":>8} {"FP":>8} {"FN":>8} {"TP":>8} {"Precision":>12} {"Recall":>10} {"F1":>8}')
print('-' * 80)

for t in thresholds:
    # Apply threshold: predict Approved if probability >= t
    y_pred_t = (y_pred_proba >= t).astype(int)
    cm_t = confusion_matrix(y_test, y_pred_t)
    tn_t, fp_t, fn_t, tp_t = cm_t[0,0], cm_t[0,1], cm_t[1,0], cm_t[1,1]

    prec_t = tp_t / (tp_t + fp_t) if (tp_t + fp_t) > 0 else 0
    rec_t  = tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0
    f1_t   = 2*prec_t*rec_t/(prec_t+rec_t) if (prec_t+rec_t) > 0 else 0

    marker = ' <-- default' if t == 0.50 else ''
    print(f'{t:>10.2f} {tn_t:>8} {fp_t:>8} {fn_t:>8} {tp_t:>8} '
          f'{prec_t:>12.4f} {rec_t:>10.4f} {f1_t:>8.4f}{marker}')

    threshold_results.append({
        'threshold': t, 'FP': fp_t, 'FN': fn_t, 'precision': prec_t,
        'recall': rec_t, 'f1': f1_t
    })

print()
print('Observation: Raising the threshold reduces FP (fewer risky approvals)')
print('but increases FN (more good applicants wrongly declined).')
print('The business must decide which error is more costly.')

## Step 8 Complete — Confusion Matrix Summary

**The 4 cells and their business meaning:**

| Cell | Count | Business Meaning |
|------|-------|----------------|
| TN | see above | Bank correctly declined a risky applicant — avoided a loss |
| FP | see above | Bank approved someone who will default — credit loss |
| FN | see above | Bank declined someone who would have repaid — lost revenue |
| TP | see above | Bank correctly approved a good applicant — interest income |

**Key lesson:** In credit risk, FP is typically more costly than FN.
The 0.5 threshold can be raised to reduce FP at the cost of more FN.
The optimal threshold depends on the estimated dollar cost of each error type.

**Next:** `09_Model_Evaluation.ipynb` — AUC, KS, Gini, SHAP, feature importance, scorecard banding
